# 05 · Attention audit across training seeds (ResNet-50)

Retrains ResNet-50 with seeds 7 and 123 on the seed-42 split (`colab/split_assignment.json`), so the test set is the same image by image, and repeats the attention audit of notebook 03 on each model.

- Training recipe copied from `CodigoTFG_DanielOrtiz.py` (the canonical run): graduated augmentation with four levels (level 3 includes `ImageCompression`), MEL boosted to level 2, inverse-frequency weighted sampler, focal loss γ = 1 with label smoothing 0.1, Adam 1e-4 / weight decay 1e-4, batch 32, at most 35 epochs, early stopping on 0.85·BACC + 0.15/(1 + loss) with patience 10 and min delta 0.001, ReduceLROnPlateau (factor 0.5, patience 4) on the negated score, gradient clipping at 1.0, cuDNN deterministic.
- `experiments/attention_supervision/attention_supervision.ipynb` is not reused: its level-3 augmentation lacks `ImageCompression`.
- Audit: Grad-CAM at `layer4[-1].conv3` (protocol of notebook 03) and at `layer4`, each model's own prediction, 1991 test images with a mask.

Runs on Colab with GPU. Writes only under `colab_seeds_resnet50/`; `colab/` is read only. Training resumes from the last epoch after a disconnect.

Outputs: per seed `resnet50_best.pth`, `history.json`, `meta.json`, `gradcam_conv3.json`, `gradcam_layer4.json`; at the top level `summary_seeds.json`.

In [ ]:
# 1 · Setup: Drive, split, labels, images and masks; local copy of the images for training.

import os, glob, json, csv, shutil, subprocess, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/ISIC2018_Task3"
COLAB_DIR = os.path.join(DRIVE_ROOT, "colab")                     # reference run, read only
MASKS_DIR = os.path.join(DRIVE_ROOT, "HAM10000_segmentations_lesion_tschandl")
OUT_ROOT = os.path.join(DRIVE_ROOT, "colab_seeds_resnet50")
CACHE_DIR = "/content/local_cache_seeds"
REPO_DIR = "/content/skin-lesion-classifier-CNN"
REPO_URL = "https://github.com/daorre1202/skin-lesion-classifier-CNN.git"

SEEDS = [7, 123]
CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}


def _mount_drive():
    if not os.path.isdir("/content/drive"):
        from google.colab import drive
        drive.mount('/content/drive')
    assert os.path.isdir(DRIVE_ROOT), f"{DRIVE_ROOT} not found"


def _exactly_one(filename, root, recursive=True):
    # stop if missing or ambiguous; never pick a match silently
    pat = os.path.join(root, "**", filename) if recursive else os.path.join(root, filename)
    hits = sorted(glob.glob(pat, recursive=recursive))
    assert hits, f"'{filename}' not found under {root}"
    assert len(hits) == 1, f"'{filename}' found {len(hits)} times under {root}: {hits}"
    return hits[0]


def _load_label_map():
    # Groundtruth/ is the folder used by notebooks 03 and 04
    label_map = {}
    for name in ["ISIC2018_Task3_Training_GroundTruth.csv",
                 "ISIC2018_Task3_Validation_GroundTruth.csv",
                 "ISIC2018_Task3_Test_GroundTruth.csv"]:
        path = os.path.join(DRIVE_ROOT, "Groundtruth", name)
        assert os.path.isfile(path), f"{path} not found"
        with open(path, newline='') as f:
            for row in csv.DictReader(f):
                onehot = {c: float(row[c]) for c in CLASSES}
                label_map[row['image']] = max(onehot, key=onehot.get)
    return label_map


def _index_raw_images():
    # same rule as notebooks 03 and 04: first occurrence per id, sorted
    per_id = {}
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        for path in glob.glob(os.path.join(DRIVE_ROOT, "**", ext), recursive=True):
            img_id = os.path.splitext(os.path.basename(path))[0]
            if img_id not in per_id:
                per_id[img_id] = path
    return dict(sorted(per_id.items()))


def _build_mask_index(masks_dir):
    idx = {}
    for p in glob.glob(os.path.join(masks_dir, "**", "*.png"), recursive=True):
        base = os.path.splitext(os.path.basename(p))[0]
        for suf in ("_segmentation", "_Segmentation", "_seg", "_mask", "_Mask"):
            if base.endswith(suf):
                base = base[: -len(suf)]
                break
        idx[base] = p
    return idx


print("1) Drive"); _mount_drive()
os.makedirs(OUT_ROOT, exist_ok=True)

print("2) repository")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
labels_repo = np.loadtxt(os.path.join(REPO_DIR, "results/seed_42/tta_labels.csv"), dtype=int)
flags = pd.read_csv(os.path.join(REPO_DIR, "results/lesion_overlap/flags.csv"))
test_flags = flags[flags.split == 'test']
TRUE_REPO = {r.image_id: CLASSES[labels_repo[r.position]] for r in test_flags.itertuples()}
NO_MATE = set(test_flags[(test_flags.has_lesion_id == 1) & (test_flags.overlap == 0)].image_id)
assert len(NO_MATE) == 1212, len(NO_MATE)

print("3) split, labels, images, masks")
SPLIT_PATH = _exactly_one("split_assignment.json", COLAB_DIR, recursive=False)
split_assignment = json.load(open(SPLIT_PATH))
_counts = dict(Counter(split_assignment.values()))
assert _counts == {'train': 7030, 'val': 2342, 'test': 2348}, _counts
label_map = _load_label_map()
id2path = _index_raw_images()
mask_index = _build_mask_index(MASKS_DIR)
ids = {sp: sorted(i for i, s in split_assignment.items() if s == sp) for sp in ('train', 'val', 'test')}
for sp, l in ids.items():
    missing = [i for i in l if i not in id2path or i not in label_map]
    assert not missing, f"{sp}: {len(missing)} ids without image or label, e.g. {missing[:3]}"
assert set(TRUE_REPO) == set(ids['test']) and all(TRUE_REPO[i] == label_map[i] for i in ids['test']), \
    "test ids or labels differ from the repository arrays"
print(f"  [ok] {_counts}; labels match the repository on all 2348 test images")

print("4) local copy of the images (once per session)")
os.makedirs(CACHE_DIR, exist_ok=True)


def _copy(iid):
    src = id2path[iid]
    dst = os.path.join(CACHE_DIR, iid + os.path.splitext(src)[1])
    if not os.path.isfile(dst):
        shutil.copy2(src, dst)
    return iid, dst


t0 = time.time()
all_ids = ids['train'] + ids['val'] + ids['test']
with ThreadPoolExecutor(max_workers=12) as ex:
    local = dict(ex.map(_copy, all_ids))
records = {sp: [(i, local[i], label_map[i]) for i in l] for sp, l in ids.items()}
masked_test = [(i, p, c) for i, p, c in records['test'] if i in mask_index]
assert len(masked_test) == 1991, len(masked_test)
print(f"  [ok] {len(local)} images in {time.time() - t0:.0f}s; {len(masked_test)} test images with mask")

1) Drive
2) repository
3) split, labels, images, masks
  [ok] {'train': 7030, 'val': 2342, 'test': 2348}; labels match the repository on all 2348 test images
4) local copy of the images (once per session)
  [ok] 11720 images in 0s; 1991 test images with mask


In [ ]:
# 2 · Definitions copied from CodigoTFG_DanielOrtiz.py (ResNet-50 settings) and the Grad-CAM of notebook 03.

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import balanced_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', "GPU runtime required"

IMG_SIZE, BATCH_SIZE, LR, WEIGHT_DECAY = 224, 32, 1e-4, 1e-4
FOCAL_GAMMA, LABEL_SMOOTHING = 1.0, 0.1
MAX_EPOCHS, ES_PATIENCE, ES_MIN_DELTA, ES_BACC_WEIGHT = 35, 10, 0.001, 0.85
SCHED_PATIENCE, SCHED_FACTOR = 4, 0.5
AUG_THR_LIGHT, AUG_THR_MEDIUM, AUG_THR_HEAVY = 2.0, 6.0, 15.0
CLINICAL_AUG_BOOST = {'MEL': 2}
NUM_WORKERS = 2
IOU_THR = 0.5

_NORM = [A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()]


def build_transforms(img_size):
    aug0 = A.Compose([A.Resize(img_size, img_size), *_NORM])
    aug1 = A.Compose([A.Resize(img_size, img_size),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=30, p=0.6), *_NORM])
    aug2 = A.Compose([A.Resize(img_size, img_size),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=90, p=0.7),
                      A.Affine(translate_percent={"x": 0.05, "y": 0.05},
                               scale=(0.9, 1.1), p=0.4), *_NORM])
    aug3 = A.Compose([A.Resize(img_size, img_size),
                      A.Affine(translate_percent={"x": 0.1, "y": 0.1},
                               scale=(0.9, 1.1), rotate=(-20, 20), p=0.5),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=180, p=0.7),
                      A.GaussNoise(std_range=(0.02, 0.10), p=0.3),
                      A.MotionBlur(blur_limit=3, p=0.2),
                      A.MedianBlur(blur_limit=3, p=0.1),
                      A.ImageCompression(quality_range=(75, 100), p=0.3), *_NORM])
    return {'aug': [aug0, aug1, aug2, aug3], 'val': aug0}


TF = build_transforms(IMG_SIZE)


class SkinDataset(Dataset):
    # same level rule as the canonical SkinDataset, built from records instead of class folders
    def __init__(self, recs, is_train):
        self.samples = list(recs)
        self.cls2idx = {c: i for i, c in enumerate(CLASSES)}
        self.is_train = is_train
        self.class_aug_level = {c: 0 for c in CLASSES}
        if is_train:
            counts = Counter(c for _, _, c in self.samples)
            mx = max(counts.values())
            for c, n in counts.items():
                r = mx / n
                self.class_aug_level[c] = (0 if r < AUG_THR_LIGHT else 1 if r < AUG_THR_MEDIUM
                                           else 2 if r < AUG_THR_HEAVY else 3)
            for c, lvl in CLINICAL_AUG_BOOST.items():
                self.class_aug_level[c] = max(self.class_aug_level[c], lvl)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, k):
        _, path, c = self.samples[k]
        img = np.array(Image.open(path).convert('RGB'))
        t = TF['aug'][self.class_aug_level[c]] if self.is_train else TF['val']
        return t(image=img)['image'], self.cls2idx[c]


class FocalLoss(nn.Module):
    def __init__(self, gamma=1.0, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma; self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        C = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(logits, self.label_smoothing / (C - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        log_probs = F.log_softmax(logits, dim=1)
        ce = -(smooth * log_probs).sum(dim=1)
        with torch.no_grad():
            p_t = log_probs.exp().gather(1, targets.unsqueeze(1)).squeeze(1)
        return ((1.0 - p_t) ** self.gamma * ce).mean()


class EarlyStopping:
    def __init__(self, patience, min_delta, checkpoint_path, bacc_weight):
        self.patience = patience; self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path; self.bacc_weight = bacc_weight
        self.best_score = -np.inf; self.best_bacc = 0.0; self.best_loss = np.inf
        self.counter = 0; self.best_epoch = 0

    def _score(self, val_bacc, val_loss):
        return self.bacc_weight * val_bacc + (1 - self.bacc_weight) / (1 + val_loss)

    def step(self, val_bacc, val_loss, model, epoch):
        score = self._score(val_bacc, val_loss)
        if score > self.best_score + self.min_delta:
            self.best_score = score; self.best_bacc = val_bacc; self.best_loss = val_loss
            self.counter = 0; self.best_epoch = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience

    def state_dict(self):
        return {k: getattr(self, k) for k in ('best_score', 'best_bacc', 'best_loss', 'counter', 'best_epoch')}

    def load_state_dict(self, d):
        for k, v in d.items():
            setattr(self, k, v)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def build_weighted_sampler(ds):
    counts = Counter(c for _, _, c in ds.samples)
    w = torch.tensor([1.0 / counts[c] for _, _, c in ds.samples], dtype=torch.float)
    return WeightedRandomSampler(w, len(w), replacement=True)


def train_one_epoch(model, loader, crit, opt):
    model.train()
    run = tot = 0; pa, la = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x); loss = crit(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        run += loss.item() * x.size(0); tot += x.size(0)
        pa += logits.argmax(1).tolist(); la += y.tolist()
    return run / tot, balanced_accuracy_score(la, pa)


def evaluate(model, loader, crit):
    model.eval()
    run = tot = 0; pa, la = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            run += crit(logits, y).item() * x.size(0); tot += x.size(0)
            pa += logits.argmax(1).tolist(); la += y.tolist()
    return run / tot, balanced_accuracy_score(la, pa), pa


# Grad-CAM exactly as notebooks 03 and 04
LAYERS = {'conv3': lambda m: m.layer4[-1].conv3, 'block': lambda m: m.layer4}


def gradcams(model, x):
    store, handles = {}, []
    for key, get in LAYERS.items():
        def _fh(mod, i, o, key=key):
            store[key] = {'a': o.detach().cpu()}
            o.register_hook(lambda g, key=key: store[key].__setitem__('g', g.detach().cpu()))
        handles.append(get(model).register_forward_hook(_fh))
    model.zero_grad()
    out = model(x.unsqueeze(0).to(device))
    pred = int(out.argmax(1).item())
    out[0, pred].backward()
    for h in handles:
        h.remove()
    cams = {}
    for key, d in store.items():
        g = d['g'].squeeze(0).numpy(); a = d['a'].squeeze(0).numpy()
        cam = np.maximum(np.einsum('c,chw->hw', g.mean(axis=(1, 2)), a), 0)
        mn, mx = cam.min(), cam.max()
        if mx - mn > 1e-8:
            cam = (cam - mn) / (mx - mn)
        cams[key] = cam
    return cams, pred


def upsample_cam(cam, size):
    im = Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    return np.array(im).astype(np.float32) / 255.0


def load_mask(mask_path, size):
    return np.array(Image.open(mask_path).convert('L').resize((size, size), Image.NEAREST)) > 127


def overlap(cam, mask):
    eps = 1e-8
    ep = float((cam * mask).sum() / (cam.sum() + eps))
    peak = np.unravel_index(int(cam.argmax()), cam.shape)
    camb = cam >= IOU_THR
    union = np.logical_or(camb, mask).sum()
    iou = float(np.logical_and(camb, mask).sum() / (union + eps)) if union > 0 else float('nan')
    return ep, bool(mask[peak]), iou


print(f"[ok] definitions loaded, device {device}")

[ok] definitions loaded, device cuda


In [ ]:
# 3 · Training, one seed after the other. Resumable: full state saved to Drive after every epoch.
# After a disconnect: run cells 1 and 2, then this cell again.

def train_seed(seed):
    run_dir = os.path.join(OUT_ROOT, f"seed{seed}")
    os.makedirs(run_dir, exist_ok=True)
    best_ckpt = os.path.join(run_dir, "resnet50_best.pth")
    state_ckpt = os.path.join(run_dir, "training_state_CHECKPOINT.pth")
    meta_path = os.path.join(run_dir, "meta.json")
    if os.path.isfile(meta_path):
        print(f"[seed {seed}] already finished: {json.load(open(meta_path))}")
        return

    set_seed(seed)
    train_ds, val_ds = SkinDataset(records['train'], True), SkinDataset(records['val'], False)
    print(f"[seed {seed}] augmentation levels: {train_ds.class_aug_level}")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=build_weighted_sampler(train_ds),
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model = model.to(device)
    crit, eval_crit = FocalLoss(FOCAL_GAMMA, LABEL_SMOOTHING), nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=SCHED_FACTOR,
                                                       patience=SCHED_PATIENCE)
    es = EarlyStopping(ES_PATIENCE, ES_MIN_DELTA, best_ckpt, ES_BACC_WEIGHT)
    history = {k: [] for k in ('train_loss', 'train_bacc', 'val_loss', 'val_bacc')}
    start = 1
    if os.path.isfile(state_ckpt):
        st = torch.load(state_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        sched.load_state_dict(st['sched']); es.load_state_dict(st['es'])
        history = st['history']; start = st['epoch'] + 1
        torch.set_rng_state(st['torch_rng'].cpu()); torch.cuda.set_rng_state_all([s.cpu() for s in st['cuda_rng']])
        np.random.set_state(st['np_rng']); random.setstate(st['py_rng'])
        print(f"[seed {seed}] resume from epoch {start} (best epoch {es.best_epoch}, val BACC {es.best_bacc:.4f})")

    t0 = time.time()
    stop = es.counter >= es.patience
    for epoch in range(start, MAX_EPOCHS + 1):
        if stop:
            break
        tr_loss, tr_bacc = train_one_epoch(model, train_loader, crit, opt)
        va_loss, va_bacc, _ = evaluate(model, val_loader, eval_crit)
        sched.step(-es._score(va_bacc, va_loss))
        for k, v in zip(history, (tr_loss, tr_bacc, va_loss, va_bacc)):
            history[k].append(v)
        stop = es.step(va_bacc, va_loss, model, epoch)
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'es': es.state_dict(), 'history': history, 'epoch': epoch,
                    'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
                    'np_rng': np.random.get_state(), 'py_rng': random.getstate()}, state_ckpt)
        print(f"  [seed {seed}] epoch {epoch:2d} | train {tr_loss:.4f} BACC {tr_bacc:.4f} | "
              f"val {va_loss:.4f} BACC {va_bacc:.4f} | {time.time() - t0:.0f}s")
        if stop:
            print(f"  [seed {seed}] early stopping, best epoch {es.best_epoch}")

    # test BACC of the selected checkpoint, single model without TTA
    model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=False))
    test_loader = DataLoader(SkinDataset(records['test'], False), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS)
    _, test_bacc, _ = evaluate(model, test_loader, eval_crit)
    json.dump(history, open(os.path.join(run_dir, "history.json"), 'w'))
    meta = {'seed': seed, 'split': SPLIT_PATH, 'best_epoch': es.best_epoch, 'val_bacc': es.best_bacc,
            'val_loss': es.best_loss, 'test_bacc': test_bacc,
            'recipe': 'CodigoTFG_DanielOrtiz.py, ResNet-50 settings'}
    json.dump(meta, open(meta_path, 'w'), indent=1)
    print(f"[seed {seed}] done: {meta}")


for s in SEEDS:
    train_seed(s)

[seed 7] augmentation levels: {'MEL': 2, 'NV': 0, 'BCC': 2, 'AKIEC': 3, 'BKL': 1, 'DF': 3, 'VASC': 3}
[seed 7] resume from epoch 8 (best epoch 6, val BACC 0.7311)
  [seed 7] epoch  8 | train 0.1825 BACC 0.9363 | val 0.6110 BACC 0.7046 | 93s
  [seed 7] epoch  9 | train 0.1694 BACC 0.9446 | val 0.4788 BACC 0.7477 | 191s
  [seed 7] epoch 10 | train 0.1541 BACC 0.9511 | val 0.4530 BACC 0.7213 | 292s
  [seed 7] epoch 11 | train 0.1674 BACC 0.9449 | val 0.5221 BACC 0.7274 | 392s
  [seed 7] epoch 12 | train 0.1439 BACC 0.9568 | val 0.4861 BACC 0.6698 | 492s
  [seed 7] epoch 13 | train 0.1439 BACC 0.9553 | val 0.4753 BACC 0.7105 | 592s
  [seed 7] epoch 14 | train 0.1371 BACC 0.9589 | val 0.4852 BACC 0.7099 | 692s
  [seed 7] epoch 15 | train 0.1063 BACC 0.9737 | val 0.4538 BACC 0.7481 | 792s
  [seed 7] epoch 16 | train 0.0912 BACC 0.9814 | val 0.4945 BACC 0.7808 | 892s
  [seed 7] epoch 17 | train 0.0883 BACC 0.9810 | val 0.4288 BACC 0.7721 | 996s
  [seed 7] epoch 18 | train 0.0867 BACC 0.9835 |

In [ ]:
# 4 · Attention audit of each seed with the protocol of notebook 03, then the comparison with seed 42.

from scipy.stats import mannwhitneyu, spearmanr

rng = np.random.default_rng(42)


def r_rb(a, b):
    return 2 * mannwhitneyu(a, b, alternative='two-sided').statistic / (len(a) * len(b)) - 1


def cmp(a, b, n_boot=2000):
    a, b = np.asarray(a), np.asarray(b)
    bs = [r_rb(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(n_boot)]
    return {'n': [len(a), len(b)], 'mean': [round(float(a.mean()), 3), round(float(b.mean()), 3)],
            'r_rb': round(float(r_rb(a, b)), 3),
            'r_rb_ci95': [round(float(v), 3) for v in np.percentile(bs, [2.5, 97.5])],
            'p': float(f"{mannwhitneyu(a, b, alternative='two-sided').pvalue:.2g}")}


def summarise(recs):
    df = pd.DataFrame(recs)
    out = {}
    for tag, s in (('all', df), ('no_lesion_mate', df[df.id.isin(NO_MATE)])):
        mal = s.true.isin(MALIGNANT)
        o = {'n': len(s), 'ep_overall': round(float(s.energy_prop.mean()), 3),
             'ep_malignant': round(float(s.energy_prop[mal].mean()), 3),
             'ep_benign': round(float(s.energy_prop[~mal].mean()), 3),
             'malignant_vs_benign': cmp(s.energy_prop[mal], s.energy_prop[~mal])}
        for c in ('MEL', 'NV'):
            t = s[s.true == c]
            if (~t.correct).sum() >= 5 and t.correct.sum() >= 5:
                o[f'{c}_error_vs_correct'] = cmp(t.energy_prop[~t.correct], t.energy_prop[t.correct])
        out[tag] = o
    return out


def audit_seed(seed):
    run_dir = os.path.join(OUT_ROOT, f"seed{seed}")
    paths = {k: os.path.join(run_dir, f"gradcam_{k}.json") for k in ('conv3', 'layer4')}
    if all(os.path.isfile(p) for p in paths.values()):
        return {k: json.load(open(p))['records'] for k, p in paths.items()}
    model = resnet50(weights=None); model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model.load_state_dict(torch.load(os.path.join(run_dir, "resnet50_best.pth"),
                                     map_location=device, weights_only=False))
    model = model.to(device).eval()
    recs = {'conv3': [], 'layer4': []}
    t0 = time.time()
    for k, (iid, path, cls) in enumerate(masked_test, start=1):
        x = TF['val'](image=np.array(Image.open(path).convert('RGB')))['image']
        cams, pred = gradcams(model, x)
        m = load_mask(mask_index[iid], IMG_SIZE)
        for key, name in (('conv3', 'conv3'), ('block', 'layer4')):
            ep, pg, iou = overlap(upsample_cam(cams[key], IMG_SIZE), m)
            recs[name].append({'id': iid, 'true': cls, 'pred': CLASSES[pred], 'correct': CLASSES[pred] == cls,
                               'energy_prop': ep, 'pointing_hit': pg, 'iou': iou, 'mask_frac': float(m.mean())})
        if k % 500 == 0:
            print(f"  [seed {seed}] {k}/{len(masked_test)} | {time.time() - t0:.0f}s")
    for name, p in paths.items():
        json.dump({'config': {'model': f'resnet50_seed{seed}', 'layer': name, 'gradcam_target': 'pred',
                              'compare_size': IMG_SIZE, 'iou_threshold': IOU_THR},
                   'records': recs[name]}, open(p, 'w'), indent=1)
    return recs


REF_CONV3 = json.load(open(_exactly_one("gradcam_segmentation_overlap.json", COLAB_DIR, recursive=False)))['records']
REF_LAYER4 = json.load(open(_exactly_one("resnet50_layer4.json", os.path.join(COLAB_DIR, "attention_backbones"),
                                         recursive=False)))['records']
per_seed = {42: {'conv3': REF_CONV3, 'layer4': REF_LAYER4}}
for s in SEEDS:
    per_seed[s] = audit_seed(s)

summary = {'seeds': {}, 'stability_vs_seed42': {}}
for s, recs in per_seed.items():
    meta = (json.load(open(os.path.join(OUT_ROOT, f"seed{s}", "meta.json"))) if s != 42
            else {'test_bacc': 0.8135, 'note': 'reference run, results/seed_42/results.json'})
    summary['seeds'][s] = {'test_bacc': meta['test_bacc'], **{k: summarise(v) for k, v in recs.items()}}
    a = summary['seeds'][s]['conv3']['all']
    print(f"seed {s:3d} | test BACC {meta['test_bacc']:.4f} | EP mal {a['ep_malignant']:.3f} ben {a['ep_benign']:.3f} "
          f"r {a['malignant_vs_benign']['r_rb']:+.3f} | "
          f"MEL err r {a.get('MEL_error_vs_correct', {}).get('r_rb', float('nan')):+.3f} | "
          f"NV err r {a.get('NV_error_vs_correct', {}).get('r_rb', float('nan')):+.3f}")
    if s != 42:
        ref = {r['id']: r['energy_prop'] for r in REF_CONV3}
        cur = {r['id']: r['energy_prop'] for r in recs['conv3']}
        common = sorted(ref)
        rho = spearmanr([ref[i] for i in common], [cur[i] for i in common]).statistic
        summary['stability_vs_seed42'][s] = {'spearman_per_image_ep_conv3': round(float(rho), 3)}
        print(f"          per-image EP vs seed 42: Spearman {rho:.3f}")

json.dump(summary, open(os.path.join(OUT_ROOT, "summary_seeds.json"), 'w'), indent=1)
print("[ok]", os.path.join(OUT_ROOT, "summary_seeds.json"))

  [seed 7] 500/1991 | 196s
  [seed 7] 1000/1991 | 391s
  [seed 7] 1500/1991 | 580s
  [seed 123] 500/1991 | 17s
  [seed 123] 1000/1991 | 34s
  [seed 123] 1500/1991 | 51s
seed  42 | test BACC 0.8135 | EP mal 0.632 ben 0.476 r +0.370 | MEL err r -0.483 | NV err r +0.412
seed   7 | test BACC 0.7858 | EP mal 0.628 ben 0.537 r +0.236 | MEL err r -0.264 | NV err r -0.023
          per-image EP vs seed 42: Spearman 0.925
seed 123 | test BACC 0.7962 | EP mal 0.640 ben 0.501 r +0.339 | MEL err r -0.604 | NV err r +0.381
          per-image EP vs seed 42: Spearman 0.955
[ok] /content/drive/MyDrive/ISIC2018_Task3/colab_seeds_resnet50/summary_seeds.json
